In [0]:
from pyspark.sql import functions as F

# load tables

CATALOG_NAME = "workspace"
SCHEMA_NAME = "metadata"

METADATA_PROFILE_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.metadata_profile"
DQ_RULES_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.dq_rule_recommendations"
DQ_RESULTS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.dq_rule_results"
DQ_SCORECARD_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.dq_scorecard_summary"

metadata_profile_df = spark.table(METADATA_PROFILE_TABLE)
dq_rules_df = spark.table(DQ_RULES_TABLE)
dq_results_df = spark.table(DQ_RESULTS_TABLE)
dq_scorecard_df = spark.table(DQ_SCORECARD_TABLE)

In [0]:
# final numbers

summary = {
    "profiled_columns": metadata_profile_df.count(),
    "pii_or_sensitive_columns": metadata_profile_df.filter(F.col("is_pii_or_sensitive") == True).count(),
    "recommended_rules": dq_rules_df.count(),
    "executed_rules": dq_results_df.count(),
    "failed_rules": dq_results_df.filter(F.col("status") == "FAIL").count(),
    "error_rules": dq_results_df.filter(F.col("status") == "ERROR").count()
}

summary

In [0]:
# governance summary

display(
    metadata_profile_df
    .groupBy("table_name", "governance_tag")
    .count()
    .orderBy("table_name", "governance_tag")
)

In [0]:
# recommended rules summary

display(
    dq_rules_df
    .groupBy("table_name", "rule_type", "severity")
    .count()
    .orderBy("table_name", "rule_type", "severity")
)

In [0]:
# data quality scorecard

display(
    dq_scorecard_df
    .orderBy(F.desc("weighted_failure_score"))
)

In [0]:
# worst failing rules

display(
    dq_results_df
    .filter(F.col("status") != "PASS")
    .select(
        "table_name",
        "col_name",
        "rule_type",
        "severity",
        "failed_records",
        "failure_rate",
        "status",
        "rule_description"
    )
    .orderBy(
        F.desc("failed_records"),
        F.desc("failure_rate")
    )
)